## Huggingface Dataset to AML Tutorial

The following notebook shows how to construct an AML pipeline that pulls a Huggingface dataset into an AML data asset 

### (Optional) Configure the environment 

In [ ]:
%set_env WORKSPACE_NAME=<WORKSPACE NAME>
%set_env RESOURCE_GROUP_NAME=<RESOURCE GROUP>
%set_env SUBSCRIPTION_ID=<SUBSCRIPTION ID>

!az configure --defaults workspace=%WORKSPACE_NAME% group=%RESOURCE_GROUP_NAME%

### Generate a job YAML 

molssiai-hub/pubchemqc-b3lyp dataset is a large collection of JSON files

In [ ]:
%%writefile pull_dataset_from_huggingface.yaml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
code: src
command: >-
  ./pull_dataset_from_huggingface.sh ${{inputs.huggingface_dataset_name}} ${{outputs.output_dir_path}}
inputs:
  huggingface_dataset_name: molssiai-hub/pubchemqc-b3lyp
outputs:
  output_dir_path:
    mode: rw_mount
    type: uri_folder
environment: 
  image: mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04:latest
compute: azureml:Standard-D48ads-v5
services:
  my_vs_code:
    type: vs_code
    nodes: all # For distributed jobs, use the `nodes` property to pick which node you want to enable interactive services on. If `nodes` are not selected, by default, interactive applications are only enabled on the head node. Values are "all", or compute node index (for ex. "0", "1" etc.)
  my_jupyter_lab:
    type: jupyter_lab
    nodes: all
display_name: pull_dataset_from_huggingface
experiment_name: pull_dataset
description: Pull dataset from huggingface to Azure Blob

### Submit AML job using the generated YAML file

In [ ]:
!az ml job create -f pull_dataset_from_huggingface.yaml

In [ ]:
%%writefile pull_dataset_from_huggingface_component.yaml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
code: src
command: >-
  python pull_dataset_from_huggingface.py ${{inputs.huggingface_dataset_name}} ${{outputs.output_dir_path}}
inputs:
  huggingface_dataset_name: 
    type: string
outputs:
  output_dir_path:
    mode: rw_mount
    type: uri_folder
environment: 
  image: mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04:latest
compute: azureml:Standard-DS3-v2
services:
  my_vs_code:
    type: vs_code
    nodes: all # For distributed jobs, use the `nodes` property to pick which node you want to enable interactive services on. If `nodes` are not selected, by default, interactive applications are only enabled on the head node. Values are "all", or compute node index (for ex. "0", "1" etc.)
  my_jupyter_lab:
    type: jupyter_lab
    nodes: all
display_name: pull_dataset_from_huggingface
name: pull_dataset_from_huggingface
experiment_name: pull_dataset
description: Pull dataset from huggingface to Azure Blob